# Examples and trade-offs

In [1]:
import time
import pandas as pd
from IPython.display import display
import ann, search, config

ann.init(rebuild=True)        # builds TF-IDF + HNSW
search.init(rebuild=False)    # exact reference over the same matrices
df = pd.read_parquet(config.PARQUET)
meta = df.set_index("uniq_id")

def show(ids, query_id=None):
    out = []
    for tag, pid in ([("query", query_id)] if query_id else []) + [("", i) for i in ids]:
        r = meta.loc[pid]
        out.append({"": tag, "category": r["category_leaf"],
                    "price": r["sales_price"], "rating": r["rating"],
                    "name": (r["product_name"] or "")[:44]})
    return pd.DataFrame(out)

HNSW built over (30000, 202)  [text=tfidf image=none numeric_w=0.3 image_w=0.5]


## 1. Baseline quality across categories

Default weights. The neighbours should stay on the query's category and gender,
that's the category text in the blob doing its job.

In [2]:
womens = meta[meta.category_leaf == "Womens Kurtas Kurtis"].index[0]
mens   = meta[meta.category_leaf == "Mens T Shirts"].index[0]
display(show(ann.find_similar_products(womens, 5), womens))
display(show(ann.find_similar_products(mens, 5), mens))

,,category,price,rating,name
0,query,Womens Kurtas Kurtis,200.0,5.0,LA' Facon Cotton Kalamkari Handblock Saree B
1,,Womens Kurtas Kurtis,200.0,5.0,LA' Facon Cotton Kalamkari Handblock Saree B
2,,Womens Sarees,449.0,5.0,Kerala cotton kasavu saree with golden borde
3,,Womens Sarees,325.0,5.0,Kanishatrendz Cotton Saree With Blouse Piece
4,,Womens Sarees,420.0,4.1,dB DESH BIDESH Women Khadi Cotton Traditiona
5,,Womens Sarees,530.0,4.2,R SELVAMANI TEX Women's Cotton Saree With Bl


,,category,price,rating,name
0,query,Mens T Shirts,265.0,3.6,Sf Jeans By Pantaloons Men's Plain Slim fit
1,,Mens T Shirts,209.0,3.4,Ajile By Pantaloons Men's Plain Slim Fit T-S
2,,Mens T Shirts,274.0,3.3,Ajile By Pantaloons Men's Plain Slim fit T-S
3,,Mens T Shirts,349.0,3.6,Urban Ranger by Pantaloons Men's Plain Slim
4,,Mens T Shirts,349.0,3.4,Urban Ranger by Pantaloons Men's Plain Slim
5,,Mens T Shirts,224.0,2.8,Ajile By Pantaloons Men's Plain Slim Fit T-S


## 2. Weight trade-off: description vs price/rating

Same query, three numeric weights. At numeric=0 neighbours are pure text/category
(closest description). As the numeric weight grows, products with a similar
price and rating are pulled up, watch the price/rating columns tighten and the
category mix loosen. This is the configurable-weighting knob, applied at query time
with no rebuild.

In [3]:
q = mens
for w in [0.0, 0.3, 1.5]:
    print(f"numeric_weight = {w}")
    display(show(ann.find_similar_products(q, 5, weights={"text":1.0,"numeric":w,"image":0})))

numeric_weight = 0.0


,,category,price,rating,name
0,,Mens T Shirts,336.0,5.0,Ajile By Pantaloons Men's Plain Slim fit T-S
1,,Mens T Shirts,274.0,3.3,Ajile By Pantaloons Men's Plain Slim fit T-S
2,,Mens T Shirts,222.0,4.5,Urban Ranger by Pantaloons Men's Plain Slim
3,,Mens T Shirts,349.0,3.4,Urban Ranger by Pantaloons Men's Plain Slim
4,,Mens T Shirts,205.0,4.3,Ajile By Pantaloons Men's Plain Slim Fit T-S


numeric_weight = 0.3


,,category,price,rating,name
0,,Mens T Shirts,209.0,3.4,Ajile By Pantaloons Men's Plain Slim Fit T-S
1,,Mens T Shirts,274.0,3.3,Ajile By Pantaloons Men's Plain Slim fit T-S
2,,Mens T Shirts,349.0,3.6,Urban Ranger by Pantaloons Men's Plain Slim
3,,Mens T Shirts,349.0,3.4,Urban Ranger by Pantaloons Men's Plain Slim
4,,Mens T Shirts,224.0,2.8,Ajile By Pantaloons Men's Plain Slim Fit T-S


numeric_weight = 1.5


,,category,price,rating,name
0,,Mens T Shirts,209.0,3.4,Ajile By Pantaloons Men's Plain Slim Fit T-S
1,,Mens T Shirts,349.0,3.6,Urban Ranger by Pantaloons Men's Plain Slim
2,,Mens T Shirts,274.0,3.3,Ajile By Pantaloons Men's Plain Slim fit T-S
3,,Mens T Shirts,349.0,3.4,Urban Ranger by Pantaloons Men's Plain Slim
4,,Mens T Shirts,224.0,2.8,Ajile By Pantaloons Men's Plain Slim Fit T-S


Low numeric weight optimises for "looks like the same product", high numeric weight
optimises for "sits in the same price/quality tier".
The default (0.3) keeps text dominant while letting price/rating break near-ties.

## 3. Exact vs ANN: speed vs exactness

search.py scans everything (exact); ann.py retrieves with HNSW then re-ranks.
They return nearly the same neighbours, but ANN is much faster, the trade-off the
Part 3 bonus is about.

In [4]:
k = 10
qs = meta.index[:200]

t = time.time()
exact = {q: search.find_similar_products(q, k) for q in qs}
exact_ms = (time.time()-t)/len(qs)*1000

t = time.time()
approx = {q: ann.find_similar_products(q, k) for q in qs}
ann_ms = (time.time()-t)/len(qs)*1000

overlap = sum(len(set(exact[q]) & set(approx[q])) for q in qs) / (k*len(qs))
print(f"top-{k} overlap (exact vs ANN): {overlap:.3f}")
print(f"exact: {exact_ms:.2f} ms/query   ANN: {ann_ms:.2f} ms/query   ({exact_ms/ann_ms:.1f}x faster)")

top-10 overlap (exact vs ANN): 0.999
exact: 11.63 ms/query   ANN: 0.47 ms/query   (24.8x faster)
